In [1]:
import pandas as pd

census = pd.read_csv('/Users/cjarjun/Desktop/cambridgeshire_dashboard/2024-2025/873_census.csv')
ks2 = pd.read_csv('/Users/cjarjun/Desktop/cambridgeshire_dashboard/2024-2025/873_ks2final.csv')

print("Census shape:", census.shape)
print("KS2 shape:", ks2.shape)

Census shape: (310, 23)
KS2 shape: (215, 311)


In [2]:
census.head(3)

,URN,LA,Estab,SCHOOLTYPE,NOR,NORG,NORB,PNORG,PNORB,TSENELSE,...,NUMEAL,NUMENGFL,NUMUNCFL,PNUMEAL,PNUMENGFL,PNUMUNCLF,NUMFSM,NUMFSMEVER,NORFSMEVER,PNUMFSMEVER
0,108886,873,6051,Independent school,49,16,33,32.65%,67.35%,37,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00%
1,110593,873,1001,State-funded nursery,57,29,28,50.88%,49.12%,1,...,27.0,30.0,0.0,47.40%,52.60%,0,0.0,NaN,NaN,0.00%
2,110594,873,1002,State-funded nursery,103,58,45,56.31%,43.69%,1,...,29.0,66.0,8.0,28.20%,64.10%,0.078,3.0,NaN,NaN,0.00%


In [3]:
ks2.head(3)

,RECTYPE,ALPHAIND,LEA,ESTAB,URN,SCHNAME,ADDRESS1,ADDRESS2,ADDRESS3,TOWN,...,PTRWM_EXP_3YR,PTRWM_HIGH_3YR,READ_AVERAGE_3YR,MAT_AVERAGE_3YR,READPROG_UNADJUSTED,WRITPROG_UNADJUSTED,MATPROG_UNADJUSTED,READPROG_DESCR,WRITPROG_DESCR,MATPROG_DESCR
0,1,3048.0,873.0,2200.0,145425.0,Bottisham Community Primary School,Beechwood Avenue,Bottisham,NaN,Cambridge,...,64%,11%,107,105,NaN,NaN,NaN,NaN,NaN,NaN
1,1,31694.0,873.0,2232.0,110698.0,Westfield Junior School,Ramsey Road,NaN,NaN,St Ives,...,47%,5%,104,102,NaN,NaN,NaN,NaN,NaN,NaN
2,1,20760.0,873.0,2239.0,110702.0,Priory Junior School,Longsands Road,NaN,NaN,St Neots,...,44%,3%,103,101,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Fix URN types so the join works
census['URN'] = census['URN'].astype(str).str.strip()
ks2['URN'] = ks2['URN'].astype('Int64').astype(str).str.strip()

# Verify they now look the same
print("Census URN sample:", census['URN'].head())
print("KS2 URN sample:", ks2['URN'].head())

Census URN sample: 0    108886
1    110593
2    110594
3    110595
4    110596
Name: URN, dtype: str
KS2 URN sample: 0    145425
1    110698
2    110702
3    138595
4    110733
Name: URN, dtype: str


In [8]:
# Check how many schools match between the two files
matched = census['URN'].isin(ks2['URN'])
print("Schools in census that match KS2:", matched.sum())
print("Schools in census with no KS2 match:", (~matched).sum())

Schools in census that match KS2: 211
Schools in census with no KS2 match: 99


In [9]:
# Merge the two datasets on URN
merged = pd.merge(census, ks2, on='URN', how='inner')

print("Merged shape:", merged.shape)
merged.head(3)

Merged shape: (211, 333)


,URN,LA,Estab,SCHOOLTYPE,NOR,NORG,NORB,PNORG,PNORB,TSENELSE,...,PTRWM_EXP_3YR,PTRWM_HIGH_3YR,READ_AVERAGE_3YR,MAT_AVERAGE_3YR,READPROG_UNADJUSTED,WRITPROG_UNADJUSTED,MATPROG_UNADJUSTED,READPROG_DESCR,WRITPROG_DESCR,MATPROG_DESCR
0,110602,873,2002,State-funded primary,365,187,178,51.23%,48.77%,9,...,38%,3%,104,102,NaN,NaN,NaN,NaN,NaN,NaN
1,110603,873,2004,State-funded primary,195,97,98,49.74%,50.26%,1,...,73%,15%,110,108,NaN,NaN,NaN,NaN,NaN,NaN
2,110604,873,2006,State-funded primary,480,227,253,47.29%,52.71%,19,...,68%,11%,107,105,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
cols = [
    'URN', 'SCHNAME', 'SCHOOLTYPE', 'PCON_NAME',
    'NOR',
    'TSENELSE',
    'TSENELK_x',
    'NUMFSM',
    'PNUMFSMEVER',
    'PTRWM_EXP',
    'PTRWM_HIGH',
    'READ_AVERAGE',
    'MAT_AVERAGE',
]

df = merged[cols].copy()
print(df.shape)
df.head(3)

(211, 13)


,URN,SCHNAME,SCHOOLTYPE,PCON_NAME,NOR,TSENELSE,TSENELK_x,NUMFSM,PNUMFSMEVER,PTRWM_EXP,PTRWM_HIGH,READ_AVERAGE,MAT_AVERAGE
0,110602,Bassingbourn Primary School,State-funded primary,South Cambridgeshire,365,9,59,74.0,20.27%,52%,7%,104,103
1,110603,Caldecote Primary School,State-funded primary,St Neots and Mid Cambridgeshire,195,1,23,28.0,15.38%,61%,6%,109,107
2,110604,Cottenham Primary School,State-funded primary,Ely and East Cambridgeshire,480,19,57,92.0,19.58%,61%,8%,107,105


In [21]:
# Fix percentage columns
pct_cols = ['PTRWM_EXP', 'PTRWM_HIGH', 'PNUMFSMEVER']
for col in pct_cols:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fix average score columns
score_cols = ['READ_AVERAGE', 'MAT_AVERAGE']
for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dtypes
df.describe()

,NOR,TSENELSE,TSENELK_x,NUMFSM,PNUMFSMEVER,PTRWM_EXP,PTRWM_HIGH,READ_AVERAGE,MAT_AVERAGE
count,211.000000,211.000000,211.000000,211.000000,211.000000,205.000000,205.000000,197.000000,197.000000
mean,251.905213,16.853081,35.691943,56.127962,23.733081,57.302439,7.273171,105.939086,104.512690
std,164.813431,33.111676,26.176050,46.618796,12.365855,19.635308,6.907903,3.139015,3.145468
min,52.000000,0.000000,0.000000,5.000000,4.740000,0.000000,0.000000,88.000000,85.000000
25%,133.000000,4.000000,20.000000,23.500000,14.775000,48.000000,2.000000,104.000000,103.000000
50%,210.000000,9.000000,29.000000,43.000000,20.490000,61.000000,6.000000,106.000000,105.000000
75%,347.500000,14.500000,47.000000,74.500000,30.930000,70.000000,11.000000,108.000000,107.000000
max,1524.000000,235.000000,177.000000,390.000000,74.740000,100.000000,33.000000,117.000000,113.000000


In [22]:
import sqlite3

# Create the database file
conn = sqlite3.connect('/Users/cjarjun/Desktop/cambridgeshire_dashboard/schools.db')

# Write the dataframe to a table called schools
df.to_sql('schools', conn, if_exists='replace', index=False)

print("Done. Rows written:", pd.read_sql('SELECT COUNT(*) FROM schools', conn).iloc[0,0])

Done. Rows written: 211


In [23]:
# Query 1 - Average attainment by constituency
query1 = """
SELECT 
    PCON_NAME,
    COUNT(*) as num_schools,
    ROUND(AVG(PTRWM_EXP), 1) as avg_expected_standard,
    ROUND(AVG(PNUMFSMEVER), 1) as avg_fsm_pct,
    ROUND(AVG(TSENELSE), 1) as avg_ehc_pupils
FROM schools
GROUP BY PCON_NAME
ORDER BY avg_expected_standard DESC
"""

result1 = pd.read_sql(query1, conn)
result1

,PCON_NAME,num_schools,avg_expected_standard,avg_fsm_pct,avg_ehc_pupils
0,North West Cambridgeshire,10,61.5,23.3,8.7
1,South Cambridgeshire,42,60.1,19.4,11.3
2,Huntingdon,34,59.7,21.1,17.4
3,Cambridge,21,59.7,28.0,23.5
4,Ely and East Cambridgeshire,34,59.3,21.8,19.7
5,St Neots and Mid Cambridgeshire,38,55.3,21.0,17.6
6,North East Cambridgeshire,31,48.2,34.7,18.0
7,South West Norfolk,1,46.0,38.6,9.0


In [24]:
query_cambridge = """
SELECT 
    PCON_NAME,
    COUNT(*) as num_schools,
    ROUND(AVG(NOR), 0) as avg_school_size,
    ROUND(AVG(TSENELSE), 1) as avg_ehc_pupils,
    ROUND(AVG(TSENELSE * 100.0 / NOR), 1) as ehc_pct_of_roll
FROM schools
GROUP BY PCON_NAME
ORDER BY ehc_pct_of_roll DESC
"""

pd.read_sql(query_cambridge, conn)

,PCON_NAME,num_schools,avg_school_size,avg_ehc_pupils,ehc_pct_of_roll
0,Ely and East Cambridgeshire,34,253.0,19.7,12.3
1,St Neots and Mid Cambridgeshire,38,261.0,17.6,10.7
2,Huntingdon,34,240.0,17.4,9.8
3,Cambridge,21,306.0,23.5,8.7
4,North East Cambridgeshire,31,300.0,18.0,7.7
5,South West Norfolk,1,153.0,9.0,5.9
6,South Cambridgeshire,42,207.0,11.3,5.8
7,North West Cambridgeshire,10,188.0,8.7,4.4


In [25]:
# Query 2 - Schools with highest SEND need vs their attainment
query2 = """
SELECT 
    SCHNAME,
    PCON_NAME,
    TSENELSE,
    TSENELK_x,
    (TSENELSE + TSENELK_x) as total_send,
    PTRWM_EXP,
    READ_AVERAGE,
    MAT_AVERAGE
FROM schools
WHERE PTRWM_EXP IS NOT NULL
ORDER BY total_send DESC
LIMIT 15
"""

result2 = pd.read_sql(query2, conn)
result2

,SCHNAME,PCON_NAME,TSENELSE,TSENELK_x,total_send,PTRWM_EXP,READ_AVERAGE,MAT_AVERAGE
0,"Castle School, Cambridge",Cambridge,235,0,235,0.0,88.0,85.0
1,Spring Common Academy,Huntingdon,202,0,202,0.0,NaN,NaN
2,Westwood Primary School,North East Cambridgeshire,12,177,189,42.0,106.0,102.0
3,Granta School,South Cambridgeshire,182,0,182,0.0,NaN,NaN
4,Meadowgate Academy,North East Cambridgeshire,181,0,181,0.0,NaN,NaN
5,Highfield Littleport Academy,Ely and East Cambridgeshire,146,0,146,0.0,NaN,NaN
6,St Andrew's CofE Primary School,Ely and East Cambridgeshire,23,113,136,75.0,107.0,107.0
7,The Martin Bacon Academy,St Neots and Mid Cambridgeshire,136,0,136,0.0,NaN,NaN
8,Alderman Jacobs School,North East Cambridgeshire,17,115,132,66.0,103.0,103.0
9,Peckover Primary School,North East Cambridgeshire,19,109,128,45.0,105.0,103.0


In [26]:
# Check what school types exist
df['SCHOOLTYPE'].value_counts()

SCHOOLTYPE
State-funded primary           198
State-funded special school     11
State-funded secondary           2
Name: count, dtype: int64

In [27]:
# Filter to mainstream primary schools only
df_primary = df[df['SCHOOLTYPE'] == 'State-funded primary'].copy()

print("Mainstream primary schools:", len(df_primary))

Mainstream primary schools: 198


In [28]:
# Write filtered table to SQLite as a separate table
df_primary.to_sql('primary_schools', conn, if_exists='replace', index=False)
print("Done. Primary schools written to database.")

Done. Primary schools written to database.


In [29]:
query_final = """
SELECT 
    PCON_NAME,
    COUNT(*) as num_schools,
    ROUND(AVG(PTRWM_EXP), 1) as avg_expected_standard,
    ROUND(AVG(PNUMFSMEVER), 1) as avg_fsm_pct,
    ROUND(AVG(TSENELSE * 100.0 / NOR), 1) as ehc_pct_of_roll
FROM primary_schools
GROUP BY PCON_NAME
ORDER BY avg_expected_standard DESC
"""

pd.read_sql(query_final, conn)



,PCON_NAME,num_schools,avg_expected_standard,avg_fsm_pct,ehc_pct_of_roll
0,Ely and East Cambridgeshire,31,65.0,18.1,3.8
1,Cambridge,20,62.7,27.2,4.2
2,South Cambridgeshire,41,61.6,18.7,3.5
3,Huntingdon,32,61.6,19.8,4.1
4,North West Cambridgeshire,10,61.5,23.3,4.4
5,St Neots and Mid Cambridgeshire,34,60.4,19.3,3.1
6,North East Cambridgeshire,29,49.9,34.2,4.6
7,South West Norfolk,1,46.0,38.6,5.9


In [30]:
# Save the connection open for now
# Summary of what we've built:
# - census: 310 schools, pupil population data
# - ks2: 215 schools, KS2 attainment data  
# - merged: 211 schools where both datasets match
# - df_primary: 198 mainstream primary schools only (special schools removed)
# - Tables in SQLite: 'schools' (211 rows), 'primary_schools' (198 rows)
# - Database location: /Users/cjarjun/Desktop/cambridgeshire_dashboard/schools.db

print("Database tables:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

Database tables:
              name
0          schools
1  primary_schools


In [31]:
query_cte = """
WITH school_summary AS (
    SELECT 
        PCON_NAME,
        SCHNAME,
        PTRWM_EXP,
        PNUMFSMEVER,
        AVG(PTRWM_EXP) OVER (PARTITION BY PCON_NAME) as area_avg_attainment
    FROM primary_schools
    WHERE PTRWM_EXP IS NOT NULL
)
SELECT 
    PCON_NAME,
    SCHNAME,
    PTRWM_EXP as school_attainment,
    ROUND(area_avg_attainment, 1) as area_average,
    ROUND(PTRWM_EXP - area_avg_attainment, 1) as vs_area_average
FROM school_summary
ORDER BY PCON_NAME, vs_area_average DESC
"""

result_cte = pd.read_sql(query_cte, conn)
result_cte.head(20)

,PCON_NAME,SCHNAME,school_attainment,area_average,vs_area_average
0,Cambridge,St Alban's Catholic Primary School,83.0,62.7,20.3
1,Cambridge,Newnham Croft Primary School,81.0,62.7,18.3
2,Cambridge,Ridgefield Primary School,80.0,62.7,17.3
3,Cambridge,Kings Hedges Primary School,79.0,62.7,16.3
4,Cambridge,St Luke's CofE Primary School,79.0,62.7,16.3
5,Cambridge,St Matthew's Primary School,76.0,62.7,13.3
6,Cambridge,Milton Road Primary School,74.0,62.7,11.3
7,Cambridge,Park Street CofE Primary School,72.0,62.7,9.3
8,Cambridge,University of Cambridge Primary School,71.0,62.7,8.3
9,Cambridge,Trumpington Park Primary School,70.0,62.7,7.3


In [32]:
# Close the database connection
conn.close()
print("All done. Database connection closed.")

All done. Database connection closed.
